# Notebook 01: Data Extraction & Profiling
**Project:** Student Mental Health & Burnout Analysis
**Purpose:** Load the raw dataset, understand its structure, and 
document every data quality issue before any cleaning begins.
**Why this matters:** You must NEVER clean data without first 
documenting what problems exist. This notebook is your "before" 
snapshot.

We import pandas for data manipulation, 
numpy for numerical operations, matplotlib and seaborn for 
visualisation, and warnings to suppress unnecessary output.

In [ ]:
import pandas as pd          
import numpy as np           
import matplotlib.pyplot as plt  
import seaborn as sns        
import warnings              
import os                    

warnings.filterwarnings('ignore')  

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("All libraries imported successfully")

We load the raw CSV file into a pandas 
DataFrame. A DataFrame is like an Excel spreadsheet in Python. 
We immediately check the shape (rows × columns) to confirm 
the file loaded correctly.

In [ ]:
RAW_PATH = '../data/raw/student_health_raw.csv'

if not os.path.exists(RAW_PATH):
    print(f"ERROR: File not found at {RAW_PATH}")
    print("Please download the dataset from Kaggle and place it in data/raw/")
else:
    print(f"File found at: {RAW_PATH}")

df_raw = pd.read_csv(RAW_PATH, low_memory=False)

print(f"\nDataset loaded successfully!")
print(f"   Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"   Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Before doing anything else, we look at the 
first few rows to understand what the data looks like. This is 
always the first step — never skip it.

In [ ]:
print("=== FIRST 5 ROWS ===")
print(df_raw.head())

print("\n=== LAST 5 ROWS ===")
print(df_raw.tail())

print("\n=== COLUMN NAMES & DATA TYPES ===")
print(df_raw.dtypes)

In [ ]:
print("=== DATASET STRUCTURE ===")
print(f"Total rows:    {df_raw.shape[0]:,}")
print(f"Total columns: {df_raw.shape[1]}")
print(f"\nColumn list:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i:2}. {col}")

print("\n=== DETAILED INFO ===")
df_raw.info()

Missing values are cells with no data. 
We need to know exactly how many are missing in each column 
and what percentage that represents. We cannot clean what we 
haven't measured.

In [ ]:
missing_count = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)

missing_summary = pd.DataFrame({
    'Column': missing_count.index,
    'Missing Count': missing_count.values,
    'Missing %': missing_pct.values
})

missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
missing_summary = missing_summary.sort_values('Missing %', ascending=False)

if len(missing_summary) == 0:
    print("No missing values found in any column!")
else:
    print(f"Found missing values in {len(missing_summary)} columns:")
    print(missing_summary.to_string(index=False))

if len(missing_summary) > 0:
    plt.figure(figsize=(10, 5))
    plt.bar(missing_summary['Column'], 
            missing_summary['Missing %'], 
            color='#E24B4A')
    plt.title('Missing Values by Column (%)', fontsize=14, fontweight='bold')
    plt.xlabel('Column Name')
    plt.ylabel('Missing %')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('../data/processed/missing_values_chart.png', dpi=150)
    plt.show()
    print("Chart saved to data/processed/")

In [ ]:

duplicate_count = df_raw.duplicated().sum()
duplicate_pct = (duplicate_count / len(df_raw) * 100).round(2)

print(f"Duplicate rows found: {duplicate_count:,} ({duplicate_pct}% of dataset)")

if duplicate_count > 0:
    print("ACTION REQUIRED: Remove duplicates in 02_cleaning.ipynb")
    print("\nSample of duplicate rows:")
    print(df_raw[df_raw.duplicated(keep=False)].head(4))
else:
    print("No duplicate rows found")

describe() gives us the min, max, mean, 
and quartiles for every numerical column. This tells us if any 
values are outside realistic ranges — for example, sleep_hours 
of 25 is impossible and must be treated as an error.

In [ ]:
print("=== STATISTICAL SUMMARY (Numerical Columns) ===")
numerical_cols = ['age', 'study_hours_per_day', 'stress_level', 
                  'anxiety_score', 'depression_score', 'sleep_hours',
                  'social_support', 'screen_time', 'internet_usage', 
                  'financial_stress']

print(df_raw[numerical_cols].describe().round(2))

print("\n=== RANGE VALIDATION CHECKS ===")
checks = {
    'age':                  (17, 30),
    'study_hours_per_day':  (0, 18),
    'stress_level':         (0, 10),
    'anxiety_score':        (0, 10),
    'depression_score':     (0, 10),
    'sleep_hours':          (2, 12),
    'social_support':       (0, 10),
    'screen_time':          (0, 16),
    'internet_usage':       (0, 16),
    'financial_stress':     (0, 10),
}

for col, (min_val, max_val) in checks.items():
    out_of_range = df_raw[
        (df_raw[col] < min_val) | (df_raw[col] > max_val)
    ].shape[0]
    status = "NEEDS FIXING" if out_of_range > 0 else "OK"
    print(f"  {col:<25} Expected: {min_val}–{max_val}  "
          f"Out of range: {out_of_range:,}  {status}")

In [ ]:
categorical_cols = ['gender', 'academic_year', 'exam_pressure',
                    'academic_performance', 'physical_activity',
                    'family_expectation']

print("=== CATEGORICAL COLUMN VALUE COUNTS ===")
for col in categorical_cols:
    print(f"\n{col}:")
    print(df_raw[col].value_counts(dropna=False))
    unique_vals = df_raw[col].dropna().unique()
    if len(unique_vals) > 10:
        print(f"  {len(unique_vals)} unique values — possible inconsistency!")

In [ ]:

plt.figure(figsize=(12, 8))
corr_matrix = df_raw[numerical_cols].corr()

sns.heatmap(
    corr_matrix,
    annot=True,           
    fmt='.2f',            
    cmap='RdYlGn',        
    center=0,             
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — All Numerical Columns\n'
          '(Darker green = stronger positive correlation, '
          'Darker red = stronger negative)', 
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/correlation_preview.png', dpi=150)
plt.show()
print("Correlation heatmap saved")

In [ ]:
print("=" * 60)
print("  EXTRACTION SUMMARY REPORT")
print("=" * 60)
print(f"  Dataset:        student_health_raw.csv")
print(f"  Total rows:     {df_raw.shape[0]:,}")
print(f"  Total columns:  {df_raw.shape[1]}")
print(f"  Missing values: {df_raw.isnull().sum().sum():,} total cells")
print(f"  Duplicate rows: {df_raw.duplicated().sum():,}")
print(f"  Memory:         {df_raw.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print("=" * 60)
print("  NEXT STEP: Open 02_cleaning.ipynb")
print("=" * 60)